# Wall Crack Detection Using YOLO11
**Object detection | YOLO11n | Google Colab GPU**

Tujuan: mendeteksi lokasi retakan pada foto dinding dengan bounding box. Dataset ADAS menyediakan 5.169 gambar (versi 1, CC BY 4.0) dengan satu kelas; jumlah setelah audit mungkin lebih rendah. Dataset ini belum memiliki split valid/test. Notebook memvalidasi label, menghapus duplikat byte-identik, membagi data, melatih model, mengevaluasi, dan mengekspor hasil.

- [Dataset](https://universe.roboflow.com/adas-fvpqc/wall-crack-p0jqt/dataset/1)
- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/)
- [Google Colab](https://colab.research.google.com/)

Pilih **Runtime > Change runtime type > GPU** sebelum mulai. Jalankan sel berurutan. Jangan gunakan hasil model untuk menyatakan keamanan struktur bangunan.


## 1. Instalasi library dan cek GPU


In [ ]:
%pip install -q "ultralytics>=8.3,<9" "roboflow>=1.1,<2" pyyaml


In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Aktifkan GPU di Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))
ROOT = Path('/content/wall_crack')
ROOT.mkdir(exist_ok=True)
RAW = ROOT / 'raw'
DATA = ROOT / 'dataset'


## 2. Download dataset
Buat akun Roboflow, buka dataset pada tautan di atas, lalu gunakan API key akun Anda. Simpan key sebagai secret Colab bernama `ROBOFLOW_API_KEY` (aktifkan akses notebook) atau masukkan saat diminta. Jangan bagikan key pada GitHub.

Alternatif: unduh ZIP berformat YOLOv11/YOLOv8 secara manual, upload ke Colab, dan ekstrak ke `RAW`. Format label bbox YOLOv8 kompatibel dengan YOLO11 detection. Jika sudah mengekstrak ZIP, sel SDK di bawah dapat dilewati.


In [ ]:
from getpass import getpass
from roboflow import Roboflow

if not (RAW / 'data.yaml').exists():
    try:
        from google.colab import userdata
        api_key = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        api_key = getpass('Roboflow API key: ')
    if not api_key:
        raise ValueError('API key diperlukan untuk SDK.')
    rf = Roboflow(api_key=api_key)
    version = rf.workspace('adas-fvpqc').project('wall-crack-p0jqt').version(1)
    version.download('yolov8', location=str(RAW))
    del api_key
print('Dataset:', RAW)


## 3. Audit dan pembagian dataset
Split 80/10/10 dengan seed 42. Gambar byte-identik hanya dipakai sekali. Label kosong dianggap gambar tanpa objek; label hilang atau rusak harus diperiksa manual. Audit ini tidak mendeteksi semua duplikat visual, atau foto berurutan dari dinding yang sama; bila informasi lokasi tersedia, gunakan split berbasis lokasi agar evaluasi lebih kuat.


In [ ]:
import hashlib
import random
import shutil
from collections import defaultdict
from pathlib import Path
import yaml
from PIL import Image

def prepare_dataset(raw_dir, out_dir, seed=42):
    """Validasi YOLO detection, kelompokkan duplikat persis, lalu split 80/10/10."""
    raw, out = Path(raw_dir).resolve(), Path(out_dir).resolve()
    if raw == out or raw in out.parents or out in raw.parents:
        raise ValueError('Folder sumber dan hasil harus terpisah.')
    ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    images = sorted(f for f in raw.rglob('*') if f.is_file() and f.suffix.lower() in ext and f.parent.name == 'images')
    if not images:
        raise ValueError('Folder images tidak ditemukan. Periksa hasil ekspor YOLO.')
    groups = defaultdict(list)
    missing, empty, invalid = [], [], []
    for img in images:
        label = img.parent.parent / 'labels' / (img.stem + '.txt')
        if not label.exists():
            missing.append(str(img))
            continue
        try:
            with Image.open(img) as im:
                im.verify()
            rows = label.read_text(encoding='utf-8').splitlines()
            for row in rows:
                if not row.strip():
                    continue
                parts = row.split()
                if len(parts) != 5 or int(parts[0]) != 0:
                    raise ValueError('Harus 1 kelas dan 5 kolom YOLO detection')
                x,y,w,h = map(float, parts[1:])
                if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    raise ValueError('Koordinat bbox tidak valid')
            if not any(r.strip() for r in rows):
                empty.append(str(img))
            digest = hashlib.sha256(img.read_bytes()).hexdigest()
            groups[digest].append((img,label))
        except Exception as e:
            invalid.append((str(img),str(e)))
    print(f'Gambar sumber: {len(images)}; label kosong: {len(empty)}; label hilang: {len(missing)}; invalid: {len(invalid)}')
    if missing or invalid:
        print('Contoh label hilang:', missing[:5], 'Contoh invalid:', invalid[:5])
        raise ValueError('Audit label/gambar bermasalah sebelum melanjutkan. Untuk gambar negatif terverifikasi, buat file .txt kosong.')
    # Satu salinan untuk setiap hash; cegah kebocoran duplikat byte-identik.
    # Jika label duplikat berbeda, minta pemeriksaan manual.
    unique = []
    for digest, group in sorted(groups.items()):
        canonical = {tuple(sorted(row.strip() for row in lab.read_text(encoding='utf-8').splitlines() if row.strip())) for _,lab in group}
        if len(canonical) != 1:
            raise ValueError(f'Duplikat dengan anotasi berbeda: {group[0][0]}')
        unique.append((digest, *group[0]))
    random.Random(seed).shuffle(unique)
    n = len(unique)
    if n < 10:
        raise ValueError('Dataset unik terlalu kecil.')
    ntrain, nval = int(n * .8), int(n * .1)
    splits = {'train': unique[:ntrain], 'val': unique[ntrain:ntrain+nval], 'test': unique[ntrain+nval:]}
    # Folder hasil boleh dibuat ulang, tetapi hanya folder yang ditandai oleh fungsi ini.
    marker = out / '.prepared_wall_crack'
    if out.exists():
        if not marker.exists():
            raise FileExistsError(f'Folder hasil sudah ada: {out}. Pilih folder lain.')
        shutil.rmtree(out)
    out.mkdir(parents=True)
    marker.touch()
    for split, items in splits.items():
        for kind in ('images','labels'):
            (out / split / kind).mkdir(parents=True)
        for digest, img, label in items:
            shutil.copy2(img, out / split / 'images' / (digest + img.suffix.lower()))
            shutil.copy2(label, out / split / 'labels' / (digest + '.txt'))
    config = {'path': str(out), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'nc': 1, 'names': ['crack']}
    (out / 'data.yaml').write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    print(f'Duplikat persis dihapus: {len(images)-n}. Gambar unik: {n}.')
    print({key: len(items) for key,items in splits.items()})
    if n < 5000:
        print('PERHATIAN: jumlah unik di bawah 5.000. Jika minimal 5.000 gambar unik wajib, tambahkan data beranotasi yang independen sebelum training.')
    return out / 'data.yaml'


In [ ]:
data_yaml = prepare_dataset(RAW, DATA)
print(data_yaml.read_text())


## 4. Visualisasi anotasi training (cek kualitas sebelum training)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
from PIL import Image

sample = random.Random(42).sample(sorted((DATA/'train/images').iterdir()), k=6)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, path in zip(axes.flat, sample):
    im = Image.open(path).convert('RGB')
    width, height = im.size
    ax.imshow(im)
    for row in (DATA/'train/labels'/f'{path.stem}.txt').read_text().splitlines():
        if not row.strip(): continue
        _, x,y,w,h = map(float, row.split())
        ax.add_patch(patches.Rectangle(((x-w/2)*width, (y-h/2)*height), w*width, h*height, linewidth=2, edgecolor='red', facecolor='none'))
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Training YOLO11n
Transfer learning dari bobot pretrained COCO. Mulai dengan 100 epoch, batch 8, imgsz 640, patience 20. Jika CUDA out of memory, ubah batch menjadi 4 atau 2. Colab dapat terputus; simpan checkpoint ke Google Drive bila diperlukan. Untuk uji pipeline cepat, gunakan 2 epoch terlebih dahulu, lalu lakukan training final dengan 100 epoch. Early stopping dapat menghentikan training sebelum 100 epoch.


In [ ]:
EPOCHS = 100
BATCH = 8
model = YOLO('yolo11n.pt')
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=640, batch=BATCH,
            device=0, workers=2, cache=False, patience=20, seed=42,
            project=str(ROOT/'runs'), name='wall_crack_yolo11n', plots=True)
RUN = Path(model.trainer.save_dir)
BEST = RUN/'weights'/'best.pt'
print('Best weights:', BEST)


## 6. Evaluasi akhir pada test set
Gunakan bobot terbaik dari validation. Jangan gunakan test set untuk memilih hyperparameter. Precision mengukur proporsi prediksi yang benar; recall mengukur proporsi objek ground truth yang ditemukan. mAP50 dan mAP50-95 mengukur kualitas deteksi berdasarkan IoU. Nilai dihasilkan dari training Anda, bukan angka contoh.


In [ ]:
import json
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(data_yaml), split='test', imgsz=640, batch=BATCH,
                         device=0, plots=True, project=str(ROOT/'evaluation'), name='test')
summary = {'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
           'mAP50': float(metrics.box.map50), 'mAP50-95': float(metrics.box.map)}
print(json.dumps(summary, indent=2))
(RUN/'test_metrics.json').write_text(json.dumps(summary, indent=2))


In [ ]:
from IPython.display import display
for name in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    path = RUN/name
    if path.exists():
        display(Image.open(path))


## 7. Prediksi contoh gambar test


In [ ]:
examples = random.Random(7).sample(sorted((DATA/'test/images').iterdir()), k=4)
for result in best_model.predict([str(x) for x in examples], conf=.25, imgsz=640, device=0, verbose=False):
    display(Image.fromarray(result.plot()[:,:,::-1]))


## 8. Uji foto dinding sendiri (opsional)


In [ ]:
from google.colab import files
from io import BytesIO
uploaded = files.upload()
for filename, contents in uploaded.items():
    im = Image.open(BytesIO(contents)).convert('RGB')
    result = best_model.predict(im, conf=.25, imgsz=640, device=0, verbose=False)[0]
    display(Image.fromarray(result.plot()[:,:,::-1]))


'
## 9. Download model dan laporan
Unduh `best.pt` dan pindahkan ke `models/best.pt` di project Streamlit. Arsip laporan berisi metrik dan grafik hasil training serta evaluasi. Simpan file ini sebelum runtime Colab direset.


In [ ]:
import zipfile
from google.colab import files
report = ROOT/'training_report.zip'
with zipfile.ZipFile(report,'w',zipfile.ZIP_DEFLATED) as z:
    for base, prefix in [(RUN, 'training'), (Path(metrics.save_dir), 'test')]:
        for f in base.rglob('*'):
            if f.is_file() and f.suffix.lower() in {'.png','.jpg','.csv','.json','.yaml'}:
                z.write(f, str(Path(prefix)/f.relative_to(base)))
files.download(str(BEST))
files.download(str(report))
